Lanette Tyler   
ST554   
Spring 2026   
Project 3

# Fitting the Model

## Preliminary Steps: Import Modules and Functions, Create Spark Session

In [1]:
#import module 
import pandas as pd

In [97]:
#import function
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, Binarizer, OneHotEncoder, PCA, VectorAssembler
from pyspark.ml import Pipeline

In [3]:
#create spark session
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/24 20:41:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Read in Data

In [4]:
#read in the project data to a standard pandas data frame and view it
power_ml_data = pd.read_csv("https://www4.stat.ncsu.edu/online/datasets/power_ml_data.csv")
power_ml_data.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


In [8]:
#convert to a spark SQL data frame
power_ml_data = spark.createDataFrame(power_ml_data)
power_ml_data.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

## Transform Data

### Hour Column to Double Type, Response Variable Power_Zone_3 Renamed 'label'

In [9]:
#check data type of Hour column
power_ml_data

DataFrame[Temperature: double, Humidity: double, Wind_Speed: double, General_Diffuse_Flows: double, Diffuse_Flows: double, Power_Zone_1: double, Power_Zone_2: double, Power_Zone_3: double, Month: bigint, Hour: bigint]

The Hour column is read in as data type "bigint." It needs to be changed to "double" type.

In [63]:
#transform data type of Hour column to "double" type
SQLtrans = SQLTransformer(
                statement = """
                    SELECT Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, Diffuse_Flows,
                           Power_Zone_1, Power_Zone_2, Power_Zone_3 AS label, Month, CAST(HOUR AS DOUBLE) AS Hour
                    FROM __THIS__
                    """)

In [64]:
#view transformation results and object
SQLtrans.transform(power_ml_data).show(10)
SQLtrans.transform(power_ml_data)

+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|      label|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538|20240.96386|    1| 0.0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599|20131.08434|    1| 0.0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693|19668.43373|    1| 0.0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422|18899.27711|    1| 0.0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043|18442.40964|    1| 0.0|
|      5.853|    76.9|     0.081|               

DataFrame[Temperature: double, Humidity: double, Wind_Speed: double, General_Diffuse_Flows: double, Diffuse_Flows: double, Power_Zone_1: double, Power_Zone_2: double, label: double, Month: bigint, Hour: double]

The column Hour is now cast to double data type.

### Hour Column to Binary

In [65]:
#make Hour column binary, Hour < 6.5
binaryTrans = Binarizer(threshold = 6.5, inputCols = ["Hour"], 
                        outputCols = ["Daytime"])
binaryTrans.transform(
    SQLtrans.transform(power_ml_data)).show(10)

+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|      label|Month|Hour|Daytime|
+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538|20240.96386|    1| 0.0|    0.0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599|20131.08434|    1| 0.0|    0.0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693|19668.43373|    1| 0.0|    0.0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422|18899.27711|    1| 0.0|    0.0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043|18442.40964|    

### Month Column One-Hot Encoded

In [66]:
#define One Hot Encoder trnasformation/model, and view object
OHEtrans = OneHotEncoder(inputCol = "Month", outputCol = "MonthVec")
OHEtrans

OneHotEncoder_331a10fd99ee

In [67]:
OHEmodel = OHEtrans.fit(
                binaryTrans.transform(
                    SQLtrans.transform(power_ml_data)))
OHEmodel

OneHotEncoderModel: uid=OneHotEncoder_331a10fd99ee, dropLast=true, handleInvalid=error

In [68]:
#fit the model, and view object
OHEmodel = OHEtrans.fit(power_ml_data)
OHEmodel

OneHotEncoderModel: uid=OneHotEncoder_331a10fd99ee, dropLast=true, handleInvalid=error

In [69]:
#tansform data with model fit plus previous transformations, and view the object
OHEmodel.transform(
            binaryTrans.transform(
                SQLtrans.transform(power_ml_data)))

DataFrame[Temperature: double, Humidity: double, Wind_Speed: double, General_Diffuse_Flows: double, Diffuse_Flows: double, Power_Zone_1: double, Power_Zone_2: double, label: double, Month: bigint, Hour: double, Daytime: double, MonthVec: vector]

In [70]:
#apply transformations to the data set and view results
OHEmodel.transform(
            binaryTrans.transform(
                SQLtrans.transform(power_ml_data))) \
    .show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|      label|Month|Hour|Daytime|      MonthVec|
+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538|20240.96386|    1| 0.0|    0.0|(12,[1],[1.0])|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599|20131.08434|    1| 0.0|    0.0|(12,[1],[1.0])|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693|19668.43373|    1| 0.0|    0.0|(12,[1],[1.0])|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422|18899.27711|    1| 0.0|    0.0|(12,[1],[1.0])|
|     

### PCA Fit

In [78]:
#create features column for principal components analysis
PCAassembler = VectorAssembler(inputCols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"], 
                            outputCol = "PCAfeatures", handleInvalid = "keep")

In [79]:
PCAassembler.transform(
            OHEmodel.transform(
                    binaryTrans.transform(
                        SQLtrans.transform(power_ml_data)))) \
    .show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+--------------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|      label|Month|Hour|Daytime|      MonthVec|         PCAfeatures|
+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+--------------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538|20240.96386|    1| 0.0|    0.0|(12,[1],[1.0])|[6.559,73.8,0.083...|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599|20131.08434|    1| 0.0|    0.0|(12,[1],[1.0])|[6.414,74.5,0.083...|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693|19668.43373|    1| 0.0|    0.0|(12,[1],[1.0])|[6.313,74.5,0.08,...|
|      6.121|    75.0|

In [80]:
PCAtrans = PCA(k = 2, inputCol = "PCAfeatures", outputCol = "PCAresult")
PCAtrans

PCA_3368272b3227

In [81]:
PCAmodel = PCAtrans.fit(
                PCAassembler.transform(
                    OHEmodel.transform(
                        binaryTrans.transform(
                            SQLtrans.transform(power_ml_data)))))
PCAmodel

PCAModel: uid=PCA_3368272b3227, k=2

In [82]:
PCAmodel.transform(
                PCAassembler.transform(
                    OHEmodel.transform(
                        binaryTrans.transform(
                            SQLtrans.transform(power_ml_data))))) \
    .show(10)

+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+--------------------+--------------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|      label|Month|Hour|Daytime|      MonthVec|         PCAfeatures|           PCAresult|
+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+-------+--------------+--------------------+--------------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538|20240.96386|    1| 0.0|    0.0|(12,[1],[1.0])|[6.559,73.8,0.083...|[1.79440486365695...|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599|20131.08434|    1| 0.0|    0.0|(12,[1],[1.0])|[6.414,74.5,0.083...|[1.80604083009823...|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.1012

In [83]:
#view principal components matrix
PCAmodel.pc

DenseMatrix(5, 2, [-0.0095, 0.0263, -0.001, -0.9553, -0.2943, 0.0075, -0.0094, 0.0025, 0.2941, -0.9557], 0)

In [84]:
PCAmodel.explainedVariance

DenseVector([0.8839, 0.1136])

### Final Features Column for Transformed Data

In [85]:
#create features column for final assembly
assembler = VectorAssembler(inputCols = ["PCAresult", "Daytime", "Power_Zone_1", "Power_Zone_2", "MonthVec"], 
                            outputCol = "features", handleInvalid = "keep")
assembler

VectorAssembler_ab97981a4de9

In [95]:
#view results
assembler.transform(
        PCAmodel.transform(
                PCAassembler.transform(
                    OHEmodel.transform(
                        binaryTrans.transform(
                            SQLtrans.transform(power_ml_data)))))) \
    .select("features").show(10, truncate = False)

+-------------------------------------------------------------------------------------+
|features                                                                             |
+-------------------------------------------------------------------------------------+
|(17,[0,1,3,4,6],[1.7944048636569538,-0.7412746447404365,34055.6962,16128.87538,1.0]) |
|(17,[0,1,3,4,6],[1.806040830098231,-0.7108534239558387,29814.68354,19375.07599,1.0]) |
|(17,[0,1,3,4,6],[1.8102297630563897,-0.7283113191158844,29128.10127,19006.68693,1.0])|
|(17,[0,1,3,4,6],[1.7986676517408837,-0.722091303219985,28228.86076,18361.09422,1.0]) |
|(17,[0,1,3,4,6],[1.8632872016379705,-0.7323046647776464,27335.6962,17872.34043,1.0]) |
|(17,[0,1,3,4,6],[1.87820674500461,-0.7628199906979496,26624.81013,17416.41337,1.0])  |
|(17,[0,1,3,4,6],[1.9152929871795548,-0.7636928919816315,25998.98734,16993.31307,1.0])|
|(17,[0,1,3,4,6],[1.9240054080702906,-0.7645388021423688,25446.07595,16661.39818,1.0])|
|(17,[0,1,3,4,6],[1.895019303530

In [96]:
#view bottom results
assembler.transform(
        PCAmodel.transform(
                PCAassembler.transform(
                    OHEmodel.transform(
                        binaryTrans.transform(
                            SQLtrans.transform(power_ml_data)))))) \
    .select("features").tail(10)

[Row(features=SparseVector(17, {0: 1.683, 1: -0.6972, 2: 1.0, 3: 34737.6426, 4: 29132.8628})),
 Row(features=SparseVector(17, {0: 1.706, 1: -0.6905, 2: 1.0, 3: 33776.4259, 4: 28230.7456})),
 Row(features=SparseVector(17, {0: 1.7094, 1: -0.6881, 2: 1.0, 3: 33387.0722, 4: 27814.6671})),
 Row(features=SparseVector(17, {0: 1.7267, 1: -0.7132, 2: 1.0, 3: 32815.2091, 4: 27564.2835})),
 Row(features=SparseVector(17, {0: 1.7554, 1: -0.698, 2: 1.0, 3: 32158.1749, 4: 27273.3968})),
 Row(features=SparseVector(17, {0: 1.7706, 1: -0.706, 2: 1.0, 3: 31160.4563, 4: 26857.3182})),
 Row(features=SparseVector(17, {0: 1.7668, 1: -0.7022, 2: 1.0, 3: 30430.4182, 4: 26124.5781})),
 Row(features=SparseVector(17, {0: 1.7466, 1: -0.6766, 2: 1.0, 3: 29590.8745, 4: 25277.6925})),
 Row(features=SparseVector(17, {0: 1.766, 1: -0.6992, 2: 1.0, 3: 28958.1749, 4: 24692.2369})),
 Row(features=SparseVector(17, {0: 1.7939, 1: -0.7331, 2: 1.0, 3: 28349.8099, 4: 24055.2317}))]

### Pipeline

In [100]:
#create pipeline
pipeline = Pipeline(stages = [SQLtrans, binaryTrans, OHEtrans, PCAassembler, PCAtrans, assembler])
pipeline

Pipeline_a1a32131165b